# Budget-Bounded Multi-Agent Research with AgentCore Payments and OpenAI

This notebook walks through a three-agent testnet research workflow. A research lead delegates free-source discovery to a public evidence analyst, and only a premium evidence analyst can buy one application-approved x402 source. AgentCore enforces the session budget and expiry outside every model.

> AgentCore Payments is in preview. This is an educational sample, not investment advice.

![Budget-bounded paid research architecture](../images/architecture.png)

*Figure 1 - The research lead delegates public discovery, while only the premium analyst can enter the payment flow.*

## 1. Install and configure

From the repository root:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
cp .env.sample .env
```

Provision the Payment Manager, connector, and instrument with the official AgentCore Payments skill or AWS setup tutorial.

Coinbase users should complete the [canonical Coinbase CDP setup guide](../../../00-getting-started/00-setup-agentcore-payments/coinbase-cdp-setup/) before running a live payment. It covers credential settings, wallet authorization, Base Sepolia funding, SDK balance verification, and troubleshooting.

Keep wallet-provider credentials out of this notebook and repository.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")
sys.path.insert(0, str(PROJECT_ROOT))

configuration = {
    "aws_region": os.getenv("AWS_REGION", "us-east-1"),
    "bedrock_model": os.getenv("BEDROCK_OPENAI_MODEL", "openai.gpt-5.5"),
    "payment_manager": bool(os.getenv("PAYMENT_MANAGER_ARN")),
    "payment_instrument": bool(os.getenv("PAYMENT_INSTRUMENT_ID")),
    "payment_session": bool(os.getenv("PAYMENT_SESSION_ID")),
    "payment_user": bool(os.getenv("PAYMENT_USER_ID")),
    "merchant_allowlist": bool(os.getenv("PAID_RESEARCH_ALLOWED_HOSTS")),
}
configuration

The check above prints only whether each setting exists. It does not display secrets or resource identifiers.

The notebook obtains a short-lived Bedrock bearer token from the active AWS credential chain; no OpenAI API key is required.

Create a fresh per-run payment session from a terminal:

```bash
python create_payment_session.py --budget 0.25 --expiry-minutes 60
export PAYMENT_SESSION_ID=<printed-session-id>
```

The application creates this boundary before invoking the agent. The agent receives no tool that can increase the cap or extend the session.

## 2. Inspect the research request

The application supplies the research question and optionally binds one premium source. The URL is captured by the premium tool closure; it is not a model-controlled tool argument. In a real finance workload, use an approved x402 provider and review its data license.

In [ ]:
from pay_for_research import build_prompt

query = "Assess the material near-term drivers and risks for AMZN."
paid_url = os.getenv(
    "PAID_RESEARCH_URL",
    "https://x402-test.genesisblock.ai/api/market-news",
)

print(build_prompt(query, paid_url))

## 3. Build the three-agent team

The research lead receives two specialists through the OpenAI Agents SDK manager pattern. The public evidence analyst receives hosted web search when supported. The premium evidence analyst alone receives the bound x402 fetch and redacted session-status tools. `parallel_tool_calls=False` keeps delegation and purchase order easy to inspect.

For OpenAI models on Amazon Bedrock, this sample currently disables hosted web search because that endpoint rejects the `filters` field emitted by the Agents SDK. The manager and premium specialist still run.

In [ ]:
from typing import NoReturn

from bedrock_openai import configure_bedrock_openai
from pay_for_research import build_agent_team
from payment import X402PaymentClient

RUN_MODEL_LIVE = os.getenv("RUN_MODEL_LIVE", "false").lower() == "true"
RUN_PAYMENT_LIVE = os.getenv("RUN_PAYMENT_LIVE", "false").lower() == "true"


class DisabledPaymentClient:
    def fetch(self, _url: str) -> NoReturn:
        raise AssertionError("Model-only smoke test must not call the paid tool")

    def session_status(self) -> NoReturn:
        raise AssertionError("Model-only smoke test must not query payment state")


runtime = None
team = None
agent = None
if RUN_MODEL_LIVE or RUN_PAYMENT_LIVE:
    runtime = configure_bedrock_openai()
    payment_client = X402PaymentClient.from_env() if RUN_PAYMENT_LIVE else DisabledPaymentClient()
    team = build_agent_team(
        payment_client,
        approved_paid_url=paid_url,
        model=runtime.model,
        include_web_search=runtime.include_web_search,
    )
    agent = team.lead
    topology = {
        "lead_tools": [tool.name for tool in team.lead.tools],
        "public_tools": [tool.name for tool in team.public_evidence.tools],
        "premium_tools": [tool.name for tool in team.premium_evidence.tools],
    }
    print(f"Built three agents with Bedrock/{runtime.model}; web search enabled: {runtime.include_web_search}")
    print(topology)
else:
    print("Set RUN_MODEL_LIVE=true for a three-agent model smoke test.")
    print("Set RUN_PAYMENT_LIVE=true only with a funded, delegated testnet wallet.")

## 4. Run the workflow

`RUN_MODEL_LIVE=true` builds all three agents and verifies that the lead delegates to the public specialist without invoking the premium specialist. `RUN_PAYMENT_LIVE=true` lets the lead delegate to both specialists and can spend testnet USDC. Nested payment approvals propagate through the premium agent-tool to the outer run.

In [ ]:
from agents import Runner, ToolCallItem
from pay_for_research import run_research

if RUN_PAYMENT_LIVE:
    result = await run_research(query, paid_url=paid_url)
    print(result)
elif RUN_MODEL_LIVE:
    result = await Runner.run(
        agent,
        """Call research_public_evidence exactly once. Ask it to return the token
PUBLIC_SPECIALIST_OK and no other text. Do not call research_premium_evidence.
After the public specialist returns, reply with exactly PAID_RESEARCH_NOTEBOOK_OK.""",
    )
    delegated_tools = [
        item.tool_name for item in result.new_items if isinstance(item, ToolCallItem) and item.tool_name is not None
    ]
    assert delegated_tools == ["research_public_evidence"], delegated_tools
    print(result.final_output, delegated_tools)
else:
    print("Live run skipped.")

## 5. Prove the hard limit

Create another session with a budget below the endpoint price and rerun the same query:

```bash
python create_payment_session.py --budget 0.01 --expiry-minutes 15
```

The lead can still delegate the gap and the premium analyst can still request the bound source. AgentCore rejects a payment that exceeds the available amount. The expected team behavior is to report the missing evidence and narrow the conclusion, not to find another merchant or trial link.

![Public research and bounded payment workflow](../images/workflow.png)

*Figure 2 - The specialist proposes a purchase; application policy and AgentCore Payments independently permit or deny it.*

## 6. Optional human approval

For a human checkpoint before the premium analyst spends, run the CLI with `--require-payment-approval`. The OpenAI Agents SDK propagates the nested function-tool interruption through the premium agent-tool and resumes the same outer run after approval or rejection.

```bash
python pay_for_research.py "Assess AMZN" --paid-url "$PAID_RESEARCH_URL" --require-payment-approval
```

## 7. What to inspect

- Team topology: the lead has specialist tools, the public analyst has search when enabled, and only the premium analyst has payment tools.
- Agent run output: nested specialist calls, public evidence, the residual gap, premium delegation, and final synthesis.
- AgentCore Payments telemetry: payment result, amount, remaining session budget, and signing latency.
- Final brief: claim-level citations, clear paid/public evidence labels, and a paid-data ledger.

For production, add role separation, AgentCore Gateway Policy, controlled network egress, data licensing controls, spend alarms, and evals for cost per successful research brief.